# Chunking

In [ ]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from transformers import AutoTokenizer
import torch
import hashlib
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from lancedb.rerankers import ColbertReranker
import ollama
import os
import json
from tqdm.notebook import tqdm
import re
import unicodedata
import subprocess


def clean_docling_chunk_strings(chunks):
    cleaned_chunks = []
    
    for chunk in chunks:
        # 2️⃣ Normalize Unicode and replace problematic punctuation
        chunk = unicodedata.normalize("NFKD", chunk).replace("\u00A0", " ")
        chunk = chunk.translate(str.maketrans({
            "–": "-", "—": "-", "‘": "'", "’": "'", "“": '"', "”": '"'
        }))

        # 3️⃣ Remove URLs (massive tokenizers killers)
        chunk = re.sub(r"http\S+", "", chunk)

        # 4️⃣ Normalize whitespace but preserve paragraphs
        chunk = re.sub(r"[ \t]+", " ", chunk)
        chunk = re.sub(r"\n\s*\n", "\n\n", chunk)  # merge single newlines, keep double
        chunk = chunk.strip()

        cleaned_chunks.append(chunk)

    return cleaned_chunks



EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
MAX_TOKENS = 2000
OLLAMA_MODEL_NAME= "chunker_full_doc"
# CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/anthropic_control_chunks_with_metadata.json"
CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/tobacco_sliding.json"

INPUT_DIR = "smoking/policy"


converter = DocumentConverter()
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME),
    max_tokens=MAX_TOKENS # Optional, uses the max token number of the HF tokenizer by default
)
chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True #Optional, defaults to true
)

study_names = [f for f in os.listdir(INPUT_DIR) if f.endswith('.pdf')]
processed_chunks=[]
try:
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        processed_chunks = json.load(f)
except FileNotFoundError:
    print(f"No existing {CHUNKS_WITH_METADATA_FILE_NAME} file found, starting fresh.")
    

chunks_with_metadata = processed_chunks.copy()
processed_studies = set(chunk["document"] for chunk in processed_chunks)

study_names = [f for f in study_names if f not in processed_studies]
print(f"Found {len(processed_studies)} studies which are already processed.\nStudies which STILL need to be processed: {len(study_names)}:\n{study_names}...")


# Creating chunks and adding Metadata

As well as semantic context with ollama (Anthropic style)

In [ ]:
from codecarbon import EmissionsTracker

tracker_proposed = EmissionsTracker(
        project_name="tobacco_document_slice",
        measure_power_secs=1,
        output_dir="./emissions_data",
        log_level="error"
    )


with tracker_proposed:
	for source in tqdm(study_names, desc="Chunking documents..."):        
		entire_doc = ""
		doc = converter.convert(f"{INPUT_DIR}/{source}").document
		chunks = list(chunker.chunk(dl_doc=doc))
		chunks_str = [chunk.text for chunk in chunks]
		chunks_str = clean_docling_chunk_strings(chunks_str)

		# Free up CUDA memory right after we got the results from Docling, so that Ollama can use the entire GPU
		if torch.cuda.is_available():
			torch.cuda.empty_cache()

		for chunk in tqdm(chunks, desc=f"Adding context for chunks of {source[:20]}...", leave=False):    
			entire_doc = ""
			chunk_index = chunks.index(chunk)

			context_length = 16_000 # Reduce window to save memory
			context_length = context_length - 2 * MAX_TOKENS # We need to reserve space for the chunk itself (twice, the context contains the chunk itself)
			total_context_chunk_number = context_length // (MAX_TOKENS*2) # 2x, cuz before and after the chunk

			start_index_original = chunk_index - total_context_chunk_number
			start_index_truncated = max(0, start_index_original) # Avoid index out of bounds

			end_index_original = chunk_index + total_context_chunk_number
			end_index_truncated = min(len(chunks)-1, end_index_original)

			if start_index_original < 0: # We are at the start of the document, so we need to add more chunks at the end
				end_index_truncated = min(len(chunks)-1, end_index_truncated + abs(start_index_original))
			if end_index_original > len(chunks)-1: # We are at the end of the document, so we need to add more chunks at the start
				start_index_truncated = max(0, start_index_truncated - abs(end_index_original - end_index_truncated))

			for i in range(start_index_truncated, end_index_truncated + 1):
				entire_doc += " " + chunks_str[i]

			entire_doc = "FULL DOCUMENT:\n" + entire_doc
			ollama_prompt = f"CHUNK:\n{chunks_str[chunk_index]}"
			history =  [{'role': 'user', 'content': entire_doc}, {'role': 'user', 'content': ollama_prompt}]

			response = ollama.chat(
				model=OLLAMA_MODEL_NAME,
				messages=history,
				# options={
				#     'gpu_layers': 100  # use  GPU for model layers if VRAM allows
				# }
			)
			context = response['message']['content']
			# print(f"Context for chunk: {context}")
			# ---- OWN APPROACH TO CONTEXT ----
			# text_to_embed = chunks_str[chunk_index] + "\n\n" + context # We put the context AFTER the chunk to not mess up cosine similarity but still benefit keyword search for exact matches

			# ---- ANTHROPIC'S APPROACH TO CONTEXT ----
			text_to_embed = context + "\n\n" + chunks_str[chunk_index] # The context is PREPENDED to the chunk as per Anthropic's original algporithm
			# print(context)
			pages = set(
					prov.page_no
					for doc_item in chunk.meta.doc_items
					for prov in doc_item.prov
				)
			id = hashlib.sha256(chunks_str[chunk_index].encode()).hexdigest()
			chunks_with_metadata.append({'text': text_to_embed, 'original_text':chunks_str[chunk_index], 'context':context, 'document':source, 'pages':list(pages), 'id': id})
			
		# Free up ollama from GPU memory so that Docling can semantically analyze the next doc even if it's like 100 pages
		subprocess.run(["ollama", "stop", OLLAMA_MODEL_NAME], check=True)
		
		#Total runtime for scientific documents: 29m 9s for 25 documents
		#Total runtime for tobacco policy documents: ----
		#VRAM usage for tobacco: 4644 MB (ollama) + various from docling based on doc length (around 1 GB)

In [3]:
# Save the the processed chunks in case VectorDB upload goes wrong.
# Luckily since this is a notebook, if the chunking is interrupted, we can still save the partial results here.
# Append new chunks to the existing file if it exists, otherwise create it
if os.path.exists(CHUNKS_WITH_METADATA_FILE_NAME):
    print(f"Appending to existing {CHUNKS_WITH_METADATA_FILE_NAME} file.")
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    # Avoid duplicate entries by id
    existing_ids = {chunk['id'] for chunk in existing_data}
    new_chunks = [chunk for chunk in chunks_with_metadata if chunk['id'] not in existing_ids]
    chunks_with_metadata = existing_data + new_chunks

with open(CHUNKS_WITH_METADATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

print(f"Results saved to {CHUNKS_WITH_METADATA_FILE_NAME}")

Results saved to preprocessed_chunks/tobacco_sliding.json


# REORDER CONTEXT AND CHUNK

In [3]:
# CONVENIENCE STEP: Prepare the chunks with metadata file for Anthropic's original approach (context PREPENDED to chunk)
CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/chunks_with_metadata.json"
if os.path.exists(CHUNKS_WITH_METADATA_FILE_NAME):
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        chunks_with_metadata = json.load(f)

    for chunk in chunks_with_metadata:
        chunk['text'] = chunk['context'] + "\n\n" + chunk['original_text']

    with open(f"preprocessed_chunks/anthropic_chunks_with_metadata.json", "w", encoding="utf-8") as f:
        json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

# Creating Database

In [ ]:
from devtools import debug
from typing import List
registry = get_registry()
hf = registry.get("huggingface").create(name=EMBEDDING_MODEL_NAME, trust_remote_code=True, device="cuda" if torch.cuda.is_available() else "cpu")


# Define model
class MyDocument(LanceModel):
    text: str 
    vector: Vector(hf.ndims()) = hf.VectorField()
    original_text: str = hf.SourceField()
    context: str
    document: str
    pages: List[str]
    id: str  # Unique identifier for the chunk


db = lancedb.connect("./db")
db.create_table("tobacco_sliding_table", schema=MyDocument, mode="overwrite") # Uncomment this line when running this cell for the first time
table = db.open_table("tobacco_sliding_table")

# Upload in batches with progress bar
with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
    chunks_with_metadata = json.load(f)
    debug(chunks_with_metadata[0])

batch_size = 100
for i in tqdm(range(0, len(chunks_with_metadata), batch_size), desc="Uploading chunks to VectorDB"):
    batch = chunks_with_metadata[i:i+batch_size]
    table.add(batch)

table.create_scalar_index("id", replace=True) # Index based on the chunk's id, used to manually prevent duplicates

reranker = ColbertReranker()
table.create_fts_index("text", replace=True) # Used by the reranker as well as the hybrid search's BM25 index
table.wait_for_index(["text_idx"])  # Wait for the indexing to finish